# 04 — Stage 1a: Conditional Return Decomposition

For each fund, raw monthly returns are decomposed without an intercept (Henriksson–Merton framework with the
supervisor's zero-beta extension):

R<sub>i,t</sub> = V<sub>f</sub>·r<sub>f,t</sub> + V<sub>f+</sub>·r<sub>f,t</sub>·1{mkt_rf<sub>t</sub> < 0}
 + V<sub>I</sub>·R<sub>I,t</sub> + V<sub>I+</sub>·R<sub>I,t</sub>·1{mkt_rf<sub>t</sub> < 0}
 + V<sub>Z</sub>·R<sub>Z,t</sub> + V<sub>Z+</sub>·R<sub>Z,t</sub>·1{r<sub>f,t</sub> > R<sub>Z,t</sub>} + ε<sub>i,t</sub>,
 with R<sub>I,t</sub> = mkt_rf<sub>t</sub> + r<sub>f,t</sub>.

OLS fund by fund, Newey–West(12) standard errors, all returns in annualised percent (× 1200).

| | |
|---|---|
| **Input** | `panel_with_factors.parquet` (NB03) |
| **Outputs** | `stage1a_results.parquet`, `stage1a_residuals.parquet` (diagnostic only — no later stage uses them) |
| **Thesis** | Sections 4.2 and 5.2, Figure 1 |
| **Expected** | 2,138 funds; median V<sub>f</sub> = 0.223 (range −53.7 to 120.9), median V<sub>I</sub> = 0.996, median V<sub>Z</sub> = −0.054; median minimum detectable cash weight 246% of assets |

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01–NB03, then *Restart Kernel → Run All*.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from thesis_config import DATA_DIR, HAC_LAGS, MIN_OBS, assert_matches, check_paths, save_fig, save_parquet

check_paths(DATA_DIR / "panel_with_factors.parquet")
print(f"pandas {pd.__version__} | numpy {np.__version__} | statsmodels {sm.__version__}")

## 1. Regression variables

In [ ]:
df = pd.read_parquet(DATA_DIR / "panel_with_factors.parquet")
print(f"Panel: {df.shape}, {df['wficn'].nunique()} funds")
assert (df["mret"] >= -1.0).all(), "returns below -100% present"   # guard; the NAV screen in NB01 removed them

# Annualised percent
cols_to_scale = ["mret", "rf", "mkt_rf", "smb", "hml", "rmw", "cma", "mom", "liq_traded", "rz_monthly"]
df[cols_to_scale] = df[cols_to_scale] * 1200

df["RI"]            = df["mkt_rf"] + df["rf"]                     # raw market return
df["bad_month"]     = (df["mkt_rf"] < 0).astype(int)              # cash beats the market
df["cash_beats_rz"] = (df["rf"] > df["rz_monthly"]).astype(int)   # cash beats the zero-beta asset

df["rf_x_bad"]        = df["rf"]         * df["bad_month"]        # -> Vf+
df["RI_x_bad"]        = df["RI"]         * df["bad_month"]        # -> VI+
df["rz_x_cash_beats"] = df["rz_monthly"] * df["cash_beats_rz"]    # -> VZ+

n_months = df["caldt"].nunique()
print(f"Months with mkt_rf < 0: {df.groupby('caldt')['bad_month'].first().sum()} of {n_months}")
print(f"Months with rf > R_Z:   {df.groupby('caldt')['cash_beats_rz'].first().sum()} of {n_months}")

## 2. Estimation

In [ ]:
X_cols = ["rf", "rf_x_bad", "RI", "RI_x_bad", "rz_monthly", "rz_x_cash_beats"]
y_col  = "mret"
NAMES  = ["Vf", "Vf_plus", "VI", "VI_plus", "VZ", "VZ_plus"]

results, residuals = [], []
for fund, g in df.groupby("wficn", sort=False):
    fund_df = g.dropna(subset=X_cols + [y_col]).sort_values("caldt")      # time order for HAC
    if len(fund_df) < MIN_OBS:
        continue
    X, y = fund_df[X_cols], fund_df[y_col]

    # Belsley-Kuh-Welsch condition index on the column-equilibrated regressors (diagnostic only)
    cond_number = np.linalg.cond(X.values / np.linalg.norm(X.values, axis=0))

    result = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})

    # statsmodels' .rsquared is UNCENTERED without an intercept; report the centered R² as well
    r2_centered = 1 - np.sum(result.resid ** 2) / np.sum((y - y.mean()) ** 2)

    row = {"wficn": fund}
    row.update({n: result.params[c] for n, c in zip(NAMES, X_cols)})
    row.update({f"se_{n}": result.bse[c] for n, c in zip(NAMES, X_cols)})
    row.update({"t_Vf": result.tvalues["rf"], "p_Vf": result.pvalues["rf"],
                "r_squared_unc": result.rsquared, "r_squared": r2_centered,
                "cond_number": cond_number, "n_obs": len(fund_df)})
    results.append(row)

    resid_df = fund_df[["wficn", "caldt"]].copy()
    resid_df["resid_1a"] = result.resid.values
    residuals.append(resid_df)

results_df   = pd.DataFrame(results)
residuals_df = pd.concat(residuals, ignore_index=True)
print(f"Estimated {len(results_df)} funds")

save_parquet(results_df,   DATA_DIR / "stage1a_results.parquet",   index=False)
save_parquet(residuals_df, DATA_DIR / "stage1a_residuals.parquet", index=False)

## 3. Results (Section 5.2)

In [ ]:
print(results_df[NAMES].describe().round(4).to_string())

vf = results_df["Vf"]
print(f"\nMedian Vf {vf.median():.3f} | range [{vf.min():.1f}, {vf.max():.1f}] | "
      f"median VI {results_df['VI'].median():.3f} | median VZ {results_df['VZ'].median():.3f}")

assert_matches("Section 5.2: funds", len(results_df), 2_138, 0)
assert_matches("Section 5.2: median Vf, VI, VZ", [vf.median(), results_df["VI"].median(), results_df["VZ"].median()],
               [0.223, 0.996, -0.054], 3)
assert_matches("Section 5.2: range of Vf", [vf.min(), vf.max()], [-53.7, 120.9], 1)

**Figure 1 — V<sub>f</sub> dispersion and r<sub>f</sub> variance.** |V<sub>f</sub>| against the variance of r<sub>f</sub>
within each fund's estimation window, coloured by the share of the fund's months in the 2009–2016 zero-rate period.

In [ ]:
fund_rf_var     = df.groupby("wficn")["rf"].var().rename("rf_variance")
fund_zirp_share = df.groupby("wficn")["caldt"].apply(lambda d: d.dt.year.between(2009, 2016).mean()).rename("zirp_share")
plot_df = results_df.merge(fund_rf_var, on="wficn", how="left").merge(fund_zirp_share, on="wficn", how="left")

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(plot_df["rf_variance"], plot_df["Vf"].abs(), c=plot_df["zirp_share"], cmap="Reds", alpha=0.6, s=15)
ax.set_xlabel("Variance of rf within fund's regression window")
ax.set_ylabel("|Vf| (absolute coefficient magnitude)")
ax.set_title("Coefficient Explosion vs Regressor Variance", fontweight="bold")
plt.colorbar(sc, ax=ax).set_label("Share of fund's months in ZIRP era (2009–2016)")
plt.tight_layout()
save_fig(fig, "fig1_vf_vs_rf_variance")
plt.show()
print(f"corr(rf variance, |Vf|) = {plot_df['rf_variance'].corr(plot_df['Vf'].abs()):.4f}")

**Minimum detectable cash weight.** MDE = 1.96 × SE(V<sub>f</sub>): the smallest true cash weight each fund's regression
could distinguish from zero at 5%, compared with the cash weights funds actually report.

In [ ]:
mde = 1.96 * results_df["se_Vf"]
pc  = df.groupby("wficn")["per_cash_clean"].mean() / 100        # reported cash, % -> fraction (not rescaled above)

print("MDE of Vf (fraction of assets):")
print(mde.describe(percentiles=[.01, .25, .50, .75, .99]).round(3).to_string())
print(f"\nBest-identified fund: MDE = {mde.min():.2f} | median fund: MDE = {mde.median():.2f} "
      f"({mde.median()*100:.0f}% of assets)")
print(f"Reported cash weight, p1–p99 across funds: [{pc.quantile(.01):.4f}, {pc.quantile(.99):.4f}]")

print("\nFunds able to detect a true cash weight c at 5%:")
for c in [0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 1.00]:
    n = (results_df["se_Vf"] < c / 1.96).sum()
    print(f"  c = {c:>4.2f}: {n:>5d} of {len(results_df)} ({n/len(results_df)*100:5.2f}%)")

assert_matches("Section 5.2: median MDE (% of assets)", round(mde.median() * 100), 246, 0)